# Induction Head Detection

Goal: identify SantaCoder attention heads with induction-like behavior on repeated-pattern prompts.

This notebook stays close to `scripts/benchmark.py`, but it is organized for reading the research story step by step. It now covers both the original synthetic repeated-token family and the later code-shaped family, and it can also load Modal artifacts saved under `outputs/induction_heads/`.


## Setup

For local understanding, keep runs small. Heavier sweeps can be run in Colab (see `notebooks/colab/`) and artifacts saved to `outputs/induction_heads/` for inspection here.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.benchmark import (
    build_prompt_dataset,
    decode_sequence,
    load_model,
    score_heads,
)


In [ ]:
prompt_family = "synthetic"  # switch to "code" after understanding the synthetic baseline
device = "cpu"
seed = 0
n_prompts = 6 if prompt_family == "synthetic" else 4
seq_len = 8

model = load_model(device=device)
dataset = build_prompt_dataset(
    model,
    prompt_family=prompt_family,
    n_prompts=n_prompts,
    seq_len=seq_len,
    seed=seed,
)
head_scores = score_heads(model, dataset)

{
    "prompt_family": prompt_family,
    "n_prompts": n_prompts,
    "seq_len": seq_len,
    "seed": seed,
    "n_layers": model.cfg.n_layers,
    "n_heads": model.cfg.n_heads,
}


## Inspect Prompt Construction

Each example has three roles:

- `repeated`: prompt prefix where the earlier context is repeated before the query position;
- `control`: matched prompt prefix where the earlier context is replaced with a distractor;
- `detection_sequence`: explicit repeated sequence used only for measuring alignment-style attention.


In [ ]:
example = dataset[0]
{
    "label": example.label,
    "sequence_tokens": decode_sequence(model, example.sequence),
    "distractor_tokens": decode_sequence(model, example.distractor),
    "target_token": model.to_single_str_token(example.target),
    "target_text": example.target_text,
    "repeated_text": example.repeated_text,
    "control_text": example.control_text,
    "detection_text": example.detection_text,
    "align_pairs_first_8": example.align_pairs[:8],
}


## Rank Detection Heads

`head_scores[layer, head]` is only a detection signal: average attention from aligned positions in the repeated copy back to the matching positions in the first copy. A high score is interesting, but it is not yet causal evidence.


In [ ]:
rows = []
for layer in range(head_scores.shape[0]):
    for head in range(head_scores.shape[1]):
        rows.append({
            "layer": layer,
            "head": head,
            "score": float(head_scores[layer, head]),
        })

ranking = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
ranking.head(10)


## Compare Prompt Families

This cell keeps the comparison small and notebook-friendly: it recomputes only the detection ranking for both supported prompt families on one configuration each.


In [ ]:
family_configs = [
    {"prompt_family": "synthetic", "n_prompts": 6, "seq_len": 8, "seed": 0},
    {"prompt_family": "code", "n_prompts": 4, "seq_len": 8, "seed": 0},
]

family_rows = []
for cfg in family_configs:
    family_dataset = build_prompt_dataset(model, **cfg)
    family_scores = score_heads(model, family_dataset)
    flat_idx = int(family_scores.view(-1).argmax().item())
    top_layer = flat_idx // model.cfg.n_heads
    top_head = flat_idx % model.cfg.n_heads
    family_rows.append({
        **cfg,
        "top_layer": top_layer,
        "top_head": top_head,
        "top_score": float(family_scores[top_layer, top_head]),
    })

pd.DataFrame(family_rows)
